# Multi-Drug Interaction Prediction using Graph Attention Networks (GATv2)
**Academic Project Phase-II (BIS786)** | Department of IS&E, JNNCE Shivamogga

**Target:** ROC-AUC > 0.86 | PR-AUC > 0.80 | Top-5 Hit Rate > 86% | Top-10 > 95%

**Optimized for Google Colab GPU** (Tesla T4 / V100 / A100) with PyTorch AMP

> Run all cells sequentially. Downloads `model_checkpoint.pt`, `evaluation_metrics.json`, `loss_curves.png` at the end.

In [ ]:
# Cell 1: Dependency Installation
import subprocess, sys
print('Installing required packages...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch-geometric', 'pandas', 'numpy', 'scikit-learn', 'requests', 'matplotlib', 'networkx'])
print('All packages ready.')

In [ ]:
# Cell 2: Imports & Device Setup
import os, urllib.request, tarfile, requests, json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from collections import Counter

SEED = 42
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED); np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__} | Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True

In [ ]:
# Cell 3: Stanford SNAP Decagon Dataset Download
DATA_DIR = 'decagon_data'
os.makedirs(DATA_DIR, exist_ok=True)
BASE_URL = 'http://snap.stanford.edu/decagon/'
FILES = ['bio-decagon-ppi.tar.gz', 'bio-decagon-targets.tar.gz', 'bio-decagon-combo.tar.gz']

for fname in FILES:
    fpath = os.path.join(DATA_DIR, fname)
    csv_name = fname.replace('.tar.gz', '.csv')
    csv_path = os.path.join(DATA_DIR, csv_name)
    if not os.path.exists(csv_path):
        print(f'Downloading {fname}...')
        urllib.request.urlretrieve(BASE_URL + fname, fpath)
        with tarfile.open(fpath, 'r:gz') as tar:
            tar.extractall(path=DATA_DIR)
        print(f'  -> {csv_name} ready.')
    else:
        print(f'{csv_name} already exists.')

In [ ]:
# Cell 4: Build Knowledge Graph & Multi-Label Matrix
def resolve_gene_symbols(gene_ids):
    print(f'Resolving {len(gene_ids)} Gene IDs...')
    url = 'https://mygene.info/v3/query'
    gene_map = {}
    for i in range(0, len(list(gene_ids)), 1000):
        batch = list(gene_ids)[i:i+1000]
        try:
            r = requests.post(url, data={'q': ','.join(str(x) for x in batch), 'scopes': 'entrezgene', 'fields': 'symbol', 'species': 'human'}, timeout=10)
            if r.status_code == 200:
                for item in r.json():
                    if item.get('query') and item.get('symbol'):
                        gene_map[str(item['query'])] = item['symbol']
        except Exception: pass
    print(f'Resolved {len(gene_map)} symbols.')
    return gene_map

print('Reading Decagon CSV files...')
ppi_df = pd.read_csv(os.path.join(DATA_DIR, 'bio-decagon-ppi.csv'))
targets_df = pd.read_csv(os.path.join(DATA_DIR, 'bio-decagon-targets.csv'))
combo_df = pd.read_csv(os.path.join(DATA_DIR, 'bio-decagon-combo.csv'))

raw_gene_ids = {str(p) for p in set(ppi_df['Gene 1']).union(ppi_df['Gene 2']).union(targets_df['Gene'])}
gene_symbol_map = resolve_gene_symbols(raw_gene_ids)

TOP_K_SIDE_EFFECTS = 50
se_counts = Counter(combo_df['Polypharmacy Side Effect'])
top_se = [se for se, _ in se_counts.most_common(TOP_K_SIDE_EFFECTS)]
combo_df_filtered = combo_df[combo_df['Polypharmacy Side Effect'].isin(top_se)].copy()

drugs = sorted(set(targets_df['STITCH']).union(combo_df_filtered['STITCH 1']).union(combo_df_filtered['STITCH 2']))
proteins = sorted(set(ppi_df['Gene 1']).union(ppi_df['Gene 2']).union(targets_df['Gene']))
num_drugs, num_proteins = len(drugs), len(proteins)
num_nodes = num_drugs + num_proteins

drug_map = {d: i for i, d in enumerate(drugs)}
protein_map = {p: i + num_drugs for i, p in enumerate(proteins)}
drug_name_map = {drug_map[d]: d for d in drugs}
protein_name_map = {protein_map[p]: gene_symbol_map.get(str(p), f'Gene:{p}') for p in proteins}

se_df = combo_df_filtered[['Polypharmacy Side Effect', 'Side Effect Name']].drop_duplicates()
side_effect_map = dict(zip(se_df['Polypharmacy Side Effect'], se_df['Side Effect Name']))
se_to_idx = {se: i for i, se in enumerate(top_se)}

print(f'Graph: {num_drugs} Drugs, {num_proteins} Proteins | Total Nodes: {num_nodes:,}')

In [ ]:
# Cell 5: Multi-Label Matrix, Splits & Edge Index
grouped = combo_df_filtered.groupby(['STITCH 1', 'STITCH 2'])['Polypharmacy Side Effect'].apply(list).reset_index()
pairs, labels = [], []
for _, row in grouped.iterrows():
    d1, d2 = drug_map.get(row['STITCH 1']), drug_map.get(row['STITCH 2'])
    if d1 is not None and d2 is not None:
        pairs.append([d1, d2])
        label = np.zeros(TOP_K_SIDE_EFFECTS, dtype=np.float32)
        for se in row['Polypharmacy Side Effect']:
            if se in se_to_idx: label[se_to_idx[se]] = 1.0
        labels.append(label)

pairs_tensor = torch.tensor(pairs, dtype=torch.long)
labels_tensor = torch.tensor(np.array(labels), dtype=torch.float32)

# Unbiased 80/10/10 split
indices = np.arange(len(pairs))
trainval_idx, test_idx = train_test_split(indices, test_size=0.10, random_state=SEED)
train_idx, val_idx = train_test_split(trainval_idx, test_size=0.1111, random_state=SEED)
train_pairs, train_labels = pairs_tensor[train_idx], labels_tensor[train_idx]
val_pairs, val_labels = pairs_tensor[val_idx], labels_tensor[val_idx]
test_pairs, test_labels = pairs_tensor[test_idx], labels_tensor[test_idx]
print(f'Splits -> Train: {len(train_pairs):,} | Val: {len(val_pairs):,} | Test: {len(test_pairs):,}')

# Edge Index (no test edge leakage)
ppi_src = [protein_map[g] for g in ppi_df['Gene 1'] if g in protein_map]
ppi_dst = [protein_map[g] for g in ppi_df['Gene 2'] if g in protein_map]
ppi_edges = torch.tensor([ppi_src + ppi_dst, ppi_dst + ppi_src], dtype=torch.long)
dt_pairs_list = [(drug_map[d], protein_map[p]) for d, p in zip(targets_df['STITCH'], targets_df['Gene']) if d in drug_map and p in protein_map]
dt_src = [p[0] for p in dt_pairs_list]; dt_dst = [p[1] for p in dt_pairs_list]
dt_edges = torch.tensor([dt_src + dt_dst, dt_dst + dt_src], dtype=torch.long)
dd_src = train_pairs[:, 0].tolist(); dd_dst = train_pairs[:, 1].tolist()
dd_edges = torch.tensor([dd_src + dd_dst, dd_dst + dd_src], dtype=torch.long)
all_edge_index = torch.cat([ppi_edges, dt_edges, dd_edges], dim=1).to(device)
print(f'Total Knowledge Graph Edges: {all_edge_index.shape[1]:,}')

# True Negative Pairs for unbiased evaluation
known = np.zeros((num_drugs, num_drugs), dtype=bool)
pp = pairs_tensor.numpy(); known[pp[:, 0], pp[:, 1]] = True; known[pp[:, 1], pp[:, 0]] = True
neg_pool = torch.tensor(np.argwhere(~known & np.triu(np.ones_like(known), 1)), dtype=torch.long)
torch.manual_seed(SEED)
val_eval_pairs = torch.cat([val_pairs, neg_pool[torch.randint(0, len(neg_pool), (len(val_pairs),))]], dim=0)
val_eval_labels = torch.cat([val_labels, torch.zeros_like(val_labels)], dim=0).numpy()
test_eval_pairs = torch.cat([test_pairs, neg_pool[torch.randint(0, len(neg_pool), (len(test_pairs),))]], dim=0)
test_eval_labels = torch.cat([test_labels, torch.zeros_like(test_labels)], dim=0).numpy()

# Positive class weights for BCE
pos_counts = train_labels.sum(dim=0)
pos_weight = torch.clamp((train_labels.shape[0] - pos_counts) / (pos_counts + 1e-5), min=1.0, max=10.0).to(device)

In [ ]:
# Cell 6: High-Performance GATv2 Architecture with Multi-Drug Attention Pooling
class HighPerformanceDrugGAT(nn.Module):
    """
    Two-layer GATv2 with Attention Pooling (N>=2) + residual skip connections.
    Matches src/model.py DrugInteractionGAT on D: drive exactly.
    """
    def __init__(self, num_nodes, in_channels=128, hidden_channels=64, out_channels=32,
                 num_heads=4, num_side_effects=50, dropout=0.25):
        super().__init__()
        self.node_embedding = nn.Embedding(num_nodes, in_channels)
        nn.init.xavier_uniform_(self.node_embedding.weight)
        self.gat1 = GATv2Conv(in_channels, hidden_channels, heads=num_heads, dropout=0.1, concat=True)
        self.bn1 = nn.BatchNorm1d(hidden_channels * num_heads)
        self.gat2 = GATv2Conv(hidden_channels * num_heads, out_channels, heads=1, dropout=0.1, concat=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.skip_proj = nn.Linear(in_channels, out_channels)
        self.attention_pooling = nn.Sequential(nn.Linear(out_channels, 16), nn.Tanh(), nn.Linear(16, 1))
        repr_dim = out_channels * 3
        self.side_effect_predictor = nn.Sequential(nn.Linear(repr_dim, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, num_side_effects))
        self.severity_predictor = nn.Sequential(nn.Linear(repr_dim, 32), nn.ReLU(), nn.Linear(32, 3))

    def encode(self, edge_index, return_attention=False):
        x0 = self.node_embedding.weight
        r = self.skip_proj(x0)
        if return_attention:
            x, att1 = self.gat1(x0, edge_index, return_attention_weights=True)
            x = F.elu(self.bn1(x))
            x, att2 = self.gat2(x, edge_index, return_attention_weights=True)
            return self.bn2(x) + r, (att1, att2)
        x = self.gat1(x0, edge_index); x = F.elu(self.bn1(x))
        x = self.gat2(x, edge_index); return self.bn2(x) + r

    def aggregate_regimen(self, drug_embeddings):
        weights = torch.softmax(self.attention_pooling(drug_embeddings), dim=1)
        return torch.sum(drug_embeddings * weights, dim=1)

    def forward_logits(self, edge_index, drug_pairs, node_emb=None):
        if node_emb is None: node_emb = self.encode(edge_index)
        d1, d2 = node_emb[drug_pairs[:, 0]], node_emb[drug_pairs[:, 1]]
        pooled = self.aggregate_regimen(torch.stack([d1, d2], dim=1))
        feats = torch.cat([pooled, d1 * d2, (d1 - d2).abs()], dim=-1)
        return self.side_effect_predictor(feats), self.severity_predictor(feats)

    def forward(self, edge_index, drug_pairs, return_attention=False):
        if return_attention:
            node_emb, att = self.encode(edge_index, return_attention=True)
        else:
            node_emb = self.encode(edge_index)
        se_logits, sev_logits = self.forward_logits(edge_index, drug_pairs, node_emb)
        se_probs = torch.sigmoid(se_logits)
        if return_attention: return se_probs, sev_logits, att
        return se_probs, sev_logits

model = HighPerformanceDrugGAT(num_nodes=num_nodes).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model ready. Total parameters: {total_params:,}')

In [ ]:
# Cell 7: Training Loop (60 Epochs, AMP, AdamW + CosineAnnealing)
EPOCHS = 60; BATCH_SIZE = 2048; LR = 2.5e-3; WEIGHT_DECAY = 1e-5; SEV_WEIGHT = 0.5

def get_severity_labels(se_labels):
    counts = se_labels.sum(dim=1)
    severity = torch.zeros(counts.shape[0], dtype=torch.long, device=se_labels.device)
    severity[(counts >= 3) & (counts <= 5)] = 1; severity[counts >= 6] = 2
    return severity

train_pairs_d = train_pairs.to(device); train_labels_d = train_labels.to(device)
neg_pool_d = neg_pool.to(device); N_train = len(train_pairs_d)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, fused=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
bce_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
ce_loss = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler()

@torch.no_grad()
def predict_probabilities(pairs_t):
    model.eval()
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        probs, _ = model(all_edge_index, pairs_t.to(device))
    return probs.float().cpu().numpy()

def compute_metrics(y_true, y_prob, thr=0.5):
    auc_micro = roc_auc_score(y_true, y_prob, average='micro')
    valid = (y_true.sum(0) > 0) & (y_true.sum(0) < len(y_true))
    auc_macro = roc_auc_score(y_true[:, valid], y_prob[:, valid], average='macro')
    ap_micro = average_precision_score(y_true, y_prob, average='micro')
    ap_macro = average_precision_score(y_true[:, valid], y_prob[:, valid], average='macro')
    pos_mask = y_true.sum(axis=1) > 0
    top5_hit = top10_hit = 0.0
    if np.any(pos_mask):
        yt, yp = y_true[pos_mask], y_prob[pos_mask]
        top5_hit = np.mean([any(yt[i, idx] == 1 for idx in np.argsort(yp[i])[-5:]) for i in range(len(yt))])
        top10_hit = np.mean([any(yt[i, idx] == 1 for idx in np.argsort(yp[i])[-10:]) for i in range(len(yt))])
    acc = ((y_prob >= thr).astype(np.float32) == y_true).mean()
    return {'roc_auc_micro': float(auc_micro), 'roc_auc_macro': float(auc_macro),
            'pr_auc_micro': float(ap_micro), 'pr_auc_macro': float(ap_macro),
            'top5_hit_rate': float(top5_hit), 'top10_hit_rate': float(top10_hit), 'accuracy': float(acc)}

def find_optimal_threshold(y_true, y_prob):
    grid = np.arange(0.30, 0.85, 0.02)
    accs = [((y_prob >= t).astype(np.float32) == y_true).mean() for t in grid]
    return float(grid[int(np.argmax(accs))])

epoch_losses, val_aucs, val_top5s, val_accs = [], [], [], []
best_metric, best_state = 0.0, None

print('=' * 94)
print(f'{"Epoch":^6} | {"Train Loss":^10} | {"Val ROC-AUC":^12} | {"Val PR-AUC":^11} | {"Top-5 Hit Rate":^15} | {"Val Acc":^9}')
print('=' * 94)

for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(N_train, device=device)
    loss_sum = torch.zeros((), device=device); batches = 0
    for i in range(0, N_train, BATCH_SIZE):
        idx = perm[i:i + BATCH_SIZE]
        bx, by = train_pairs_d[idx], train_labels_d[idx]
        n_neg = len(bx)
        neg_idx = torch.randint(0, len(neg_pool_d), (n_neg,), device=device)
        bx = torch.cat([bx, neg_pool_d[neg_idx]], dim=0)
        by = torch.cat([by, torch.zeros(n_neg, by.shape[1], device=device)], dim=0)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            se_logits, sev_logits = model.forward_logits(all_edge_index, bx)
            loss = bce_loss(se_logits.float(), by) + SEV_WEIGHT * ce_loss(sev_logits.float(), get_severity_labels(by))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 4.0)
        scaler.step(optimizer); scaler.update()
        loss_sum += loss.detach(); batches += 1
    scheduler.step()
    avg_loss = (loss_sum / batches).item(); epoch_losses.append(avg_loss)
    if epoch % 2 == 0 or epoch == 1:
        val_probs = predict_probabilities(val_eval_pairs)
        vm = compute_metrics(val_eval_labels, val_probs)
        val_aucs.append(vm['roc_auc_micro']); val_top5s.append(vm['top5_hit_rate']); val_accs.append(vm['accuracy'])
        print(f" {epoch:02d}   | {avg_loss:10.4f} | {vm['roc_auc_micro']:12.4f} | {vm['pr_auc_micro']:11.4f} | {vm['top5_hit_rate']*100:13.2f}% | {vm['accuracy']*100:8.2f}%")
        composite = vm['roc_auc_micro'] + vm['top5_hit_rate']
        if composite > best_metric:
            best_metric = composite
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

In [ ]:
# Cell 8: Final Unbiased Test Evaluation
model.load_state_dict(best_state)
opt_thr = find_optimal_threshold(val_eval_labels, predict_probabilities(val_eval_pairs))
test_prob = predict_probabilities(test_eval_pairs)
m = compute_metrics(test_eval_labels, test_prob, thr=opt_thr)

with torch.no_grad():
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        _, sev_logits = model(all_edge_index, test_eval_pairs.to(device))
    sev_preds = torch.argmax(sev_logits, dim=1).cpu().numpy()
    sev_targets = get_severity_labels(torch.tensor(test_eval_labels)).cpu().numpy()

prec, rec, f1_sev, _ = precision_recall_fscore_support(sev_targets, sev_preds, labels=[0,1,2], zero_division=0)
conf_matrix = confusion_matrix(sev_targets, sev_preds, labels=[0,1,2])

print('=' * 94)
print('FINAL BENCHMARK TEST SET EVALUATION (IEEE Transactions Verification)')
print('=' * 94)
print(f"  Micro ROC-AUC               : {m['roc_auc_micro']:.4f}")
print(f"  Macro ROC-AUC               : {m['roc_auc_macro']:.4f}")
print(f"  Micro PR-AUC                : {m['pr_auc_micro']:.4f}")
print(f"  Macro PR-AUC                : {m['pr_auc_macro']:.4f}")
print(f"  Top-5 Diagnostic Hit Rate   : {m['top5_hit_rate']*100:.2f}%  (Target: >86%)")
print(f"  Top-10 Diagnostic Hit Rate  : {m['top10_hit_rate']*100:.2f}% (Target: >95%)")
print(f"  Calibrated Accuracy (@{opt_thr:.2f})  : {m['accuracy']*100:.2f}%")
print(f'  Severity F1: Minor={f1_sev[0]:.3f} | Moderate={f1_sev[1]:.3f} | Major={f1_sev[2]:.3f}')
print('=' * 94)

In [ ]:
# Cell 9: Save Production Checkpoint & Evaluation Artifacts
checkpoint_data = {
    'model_state': model.state_dict(),
    'drug_map': drug_map,
    'protein_map': protein_map,
    'side_effect_map': side_effect_map,
    'drug_name_map': drug_name_map,
    'protein_name_map': protein_name_map,
    'num_nodes': num_nodes,
    'num_side_effects': TOP_K_SIDE_EFFECTS,
    'all_edge_index': all_edge_index.cpu(),
    'threshold': opt_thr,
    'hyperparams': {'in_channels': 128, 'hidden_channels': 64, 'out_channels': 32, 'num_heads': 4, 'dropout': 0.25},
    'test_metrics': m,
    'severity_metrics': {'precision': prec.tolist(), 'recall': rec.tolist(), 'f1': f1_sev.tolist(), 'confusion_matrix': conf_matrix.tolist()}
}
torch.save(checkpoint_data, 'model_checkpoint.pt')
print('[SUCCESS] model_checkpoint.pt saved.')

eval_json = {
    'phase': 'Phase-2 SOTA Optimized Architecture',
    'architecture': 'GATv2 with Multi-Head Dynamic Attention & Weighted BCE',
    'ROC-AUC_macro': round(m['roc_auc_macro'], 4),
    'PR-AUC_macro': round(m['pr_auc_macro'], 4),
    'Binary_Accuracy': round(m['accuracy'], 4),
    'Top5_Retrieval_Accuracy': round(m['top5_hit_rate'], 4),
    'Top10_Retrieval_Accuracy': round(m['top10_hit_rate'], 4),
    'Confusion_Matrix': conf_matrix.tolist()
}
with open('evaluation_metrics.json', 'w') as f:
    json.dump(eval_json, f, indent=4)
print('[SUCCESS] evaluation_metrics.json saved.')

In [ ]:
# Cell 10: Publication-Quality Training Curves
plt.figure(figsize=(16, 4.2), dpi=120)
plt.subplot(1, 3, 1)
plt.plot(range(1, len(epoch_losses)+1), epoch_losses, color='#0f172a', linewidth=2.0)
plt.xlabel('Training Epoch'); plt.ylabel('Multi-Task Loss (BCE + 0.5 CE)')
plt.title('Multi-Task Convergence Curve'); plt.grid(True, linestyle='--', alpha=0.5)

plt.subplot(1, 3, 2)
eval_epochs = [1] + list(range(2, EPOCHS+1, 2))
plt.plot(eval_epochs, val_aucs, label='Val Micro ROC-AUC', color='#2563eb', linewidth=2.0)
plt.plot(eval_epochs, val_top5s, label='Top-5 Hit Rate', color='#059669', linewidth=2.0)
plt.xlabel('Epoch'); plt.ylabel('Validation Score'); plt.title('Validation Metric Progression')
plt.legend(); plt.grid(True, linestyle='--', alpha=0.5)

plt.subplot(1, 3, 3)
plt.plot(eval_epochs, [a*100 for a in val_accs], color='#d97706', linewidth=2.0)
plt.xlabel('Epoch'); plt.ylabel('Accuracy (%)'); plt.title('Diagnostic Accuracy (%) Progression')
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout(); plt.savefig('loss_curves.png'); plt.show()
print('loss_curves.png saved.')

In [ ]:
# Cell 11: Download Output Files
try:
    from google.colab import files
    print('Downloading files...')
    files.download('model_checkpoint.pt')
    files.download('evaluation_metrics.json')
    files.download('loss_curves.png')
except ImportError:
    print('Local run detected. Files saved in working directory.')